# 01 — Data Quality Check: `sets.csv` & `themes.csv`

Rebrickable LEGO veri setinin temel doğrulaması. Amaç:

1. Şema kontrolü (satır sayısı, kolonlar, dtype'lar)
2. Null oranları
3. Yıl aralığı kontrolü
4. `sets.theme_id` → `themes.id` join kontrolü (yetim kayıtlar)
5. `num_parts = 0` olan setlerin oranı (muhtemelen minifig-only / promosyon setleri — ana analizden filtrelenecek)


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RAW = "../data/raw"


## 1. Veriyi oku

In [2]:
sets = pd.read_csv(f"{RAW}/sets.csv")
themes = pd.read_csv(f"{RAW}/themes.csv")

print(f"sets:   {sets.shape[0]:,} satır, {sets.shape[1]} kolon")
print(f"themes: {themes.shape[0]:,} satır, {themes.shape[1]} kolon")


sets:   28,180 satır, 6 kolon
themes: 496 satır, 3 kolon


## 2. Şema kontrolü

In [3]:
print("--- sets.csv ---")
sets.info()


--- sets.csv ---
<class 'pandas.DataFrame'>
RangeIndex: 28180 entries, 0 to 28179
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   set_num    28180 non-null  str  
 1   name       28180 non-null  str  
 2   year       28180 non-null  int64
 3   theme_id   28180 non-null  int64
 4   num_parts  28180 non-null  int64
 5   img_url    28180 non-null  str  
dtypes: int64(3), str(3)
memory usage: 1.3 MB


In [4]:
sets.head()


,set_num,name,year,theme_id,num_parts,img_url
0,0003977811-1,Ninjago: Book of Adventures,2022,761,1,https://cdn.rebrickable.com/media/sets/0003977...
1,001-1,Gears,1965,756,43,https://cdn.rebrickable.com/media/sets/001-1.jpg
2,0011-2,Town Mini-Figures,1979,67,12,https://cdn.rebrickable.com/media/sets/0011-2.jpg
3,0011-3,Castle 2 for 1 Bonus Offer,1987,199,0,https://cdn.rebrickable.com/media/sets/0011-3.jpg
4,0012-1,Space Mini-Figures,1979,143,12,https://cdn.rebrickable.com/media/sets/0012-1.jpg


In [5]:
print("--- themes.csv ---")
themes.info()


--- themes.csv ---
<class 'pandas.DataFrame'>
RangeIndex: 496 entries, 0 to 495
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         496 non-null    int64  
 1   name       496 non-null    str    
 2   parent_id  346 non-null    float64
dtypes: float64(1), int64(1), str(1)
memory usage: 11.8 KB


In [6]:
themes.head()


,id,name,parent_id
0,1,Technic,NaN
1,3,Competition,1.0
2,4,Expert Builder,1.0
3,16,RoboRiders,1.0
4,17,Speed Slammers,1.0


## 3. Null oranları

In [7]:
def null_report(df, name):
    rep = pd.DataFrame({
        "null_count": df.isna().sum(),
        "null_ratio_%": (df.isna().mean() * 100).round(2),
    })
    rep.index.name = f"{name} column"
    return rep

null_report(sets, "sets")


,null_count,null_ratio_%
sets column,,
set_num,0,0.0
name,0,0.0
year,0,0.0
theme_id,0,0.0
num_parts,0,0.0
img_url,0,0.0


In [8]:
null_report(themes, "themes")


,null_count,null_ratio_%
themes column,,
id,0,0.00
name,0,0.00
parent_id,150,30.24


## 4. Yıl aralığı kontrolü

`sets.year` üzerinde min/max, dağılım ve makul olmayan (örn. gelecek yıl veya çok eski / 0) değer kontrolü.


In [9]:
from datetime import datetime

current_year = datetime.now().year

print(f"year min : {sets['year'].min()}")
print(f"year max : {sets['year'].max()}")
print(f"unique yıl sayısı: {sets['year'].nunique()}")
print()

future_years = sets[sets["year"] > current_year]
print(f"Bugünün yılından ({current_year}) büyük yıl içeren set sayısı: {len(future_years)}")
if len(future_years):
    display(future_years[["set_num", "name", "year"]].sort_values("year", ascending=False).head(10))


year min : 1949
year max : 2027
unique yıl sayısı: 77

Bugünün yılından (2026) büyük yıl içeren set sayısı: 24


,set_num,name,year
13041,53894-1,Watercolor Set,2027
13042,53895-1,Duplo Water Art Play Mat,2027
27238,LGLKE262H-1,Rechargeable Minifigure Key Light (All Colors),2027
25054,9781534357921-1,Ninjago: Masters of Spinjitzu,2027
24517,9780241838570-1,City: Game On: Penguin Peril,2027
24516,9780241838563-1,Ninjago: Dragons Rising: Riyu the Dragon: Nood...,2027
24515,9780241838389-1,DK Super Readers Level 1: Ninjago: Go Team Ninja,2027
24514,9780241836712-1,DK Super Readers Level 2: Marvel Super Heroes:...,2027
24513,9780241831953-1,Minecraft: Would you Rather?,2027
19537,757894517472-1,Chequered Brick Lunch Bag,2027


In [10]:
sets["year"].describe()


count    28180.000000
mean      2010.979915
std         14.031003
min       1949.000000
25%       2004.000000
50%       2015.000000
75%       2021.000000
max       2027.000000
Name: year, dtype: float64

## 5. `sets.theme_id` → `themes.id` join kontrolü

Her set'in theme_id değeri themes tablosunda karşılık bulmalı; null theme_id ve "yetim" (orphan) theme_id'leri ayrı ayrı raporlanıyor.


In [11]:
null_theme_id = sets["theme_id"].isna().sum()
print(f"Null theme_id sayısı: {null_theme_id} ({null_theme_id / len(sets) * 100:.2f}%)")

valid_theme_ids = set(themes["id"])
non_null_theme_ids = sets["theme_id"].dropna()
orphan_mask = ~non_null_theme_ids.isin(valid_theme_ids)
orphan_theme_ids = non_null_theme_ids[orphan_mask]

print(f"themes.id'de karşılığı olmayan (orphan) theme_id sayısı: {orphan_mask.sum()} "
      f"({orphan_mask.sum() / len(sets) * 100:.2f}%)")

if orphan_mask.sum():
    print("Orphan theme_id örnekleri:")
    display(sets.loc[orphan_theme_ids[orphan_mask].index, ["set_num", "name", "theme_id"]].head(10))
else:
    print("✅ Tüm non-null theme_id değerleri themes.id içinde mevcut — join sorunsuz.")


Null theme_id sayısı: 0 (0.00%)
themes.id'de karşılığı olmayan (orphan) theme_id sayısı: 0 (0.00%)
✅ Tüm non-null theme_id değerleri themes.id içinde mevcut — join sorunsuz.


## 6. `num_parts = 0` olan setler

Bu setler muhtemelen minifig-only setler, promosyon (gift-with-purchase) ürünleri veya parça sayısı hiç girilmemiş kayıtlar.
Ana analiz (ör. set büyüklüğü / karmaşıklık trendleri) bu setlerle çarpıtılacağından, ayrı ayrı raporlanıp sonraki adımlarda filtrelenmesi öneriliyor.


In [12]:
zero_parts = sets[sets["num_parts"] == 0]
zero_ratio = len(zero_parts) / len(sets) * 100

print(f"num_parts = 0 olan set sayısı: {len(zero_parts):,} / {len(sets):,}")
print(f"Oran: {zero_ratio:.2f}%")


num_parts = 0 olan set sayısı: 8,138 / 28,180
Oran: 28.88%


In [13]:
zero_parts_with_theme = zero_parts.merge(
    themes[["id", "name"]].rename(columns={"id": "theme_id", "name": "theme_name"}),
    on="theme_id", how="left",
)

print("num_parts=0 setlerin en sık geçtiği temalar (ilk 15):")
zero_parts_with_theme["theme_name"].value_counts().head(15)


num_parts=0 setlerin en sık geçtiği temalar (ilk 15):


theme_name
Bags, Totes, & Luggage            973
Key Chain                         785
Clothing & Footwear               704
Stationery and Office Supplies    674
Houseware                         614
Gear                              507
Story Books                       385
Video Games and Accessories       277
Activity Books                    253
Role Play Toys and Costumes       235
Plush Toys                        162
Storage                           138
Non-fiction Books                 130
Bag and Luggage Tags              130
Clocks and Watches                111
Name: count, dtype: int64

In [14]:
zero_parts.sample(min(10, len(zero_parts)), random_state=42)[["set_num", "name", "year", "theme_id", "num_parts"]]


,set_num,name,year,theme_id,num_parts
3106,203062508-1,Ninjago Lloyd Backpack with Gym Bag and Pencil...,2026,777,0
15763,6572607-1,Easter Stickers,2025,501,0
7250,4060-2,Multi Basket,2014,740,0
3843,238-3,LEGO System Idea Book no. 1,1962,757,0
7470,4084039-1,Marvel Avengers with Silver Centurion Minifigu...,2016,742,0
27418,LMP301A-1,Star Wars: Darth Vader on the Rebel Hunt,2019,759,0
15950,66151-1,Limited Edition Green Brick Tub Value-Pack,2006,505,0
24600,9780545649926-1,Legends of Chima: How to Draw: Heroes and Vill...,2014,760,0
12166,5010250-1,Nike x LEGO Collection Big Kids' T-Shirt - White,2026,799,0
21319,850451-1,Lord Vampyre Key Chain,2012,503,0


## 7. Özet

- **sets.csv**: satır/kolon sayısı, dtype'lar ve null oranları yukarıda raporlandı.
- **themes.csv**: satır/kolon sayısı, dtype'lar ve null oranları yukarıda raporlandı.
- **Yıl aralığı**: `sets.year` min–max değerleri ve bugünün yılından büyük anormal kayıtlar kontrol edildi.
- **Join kontrolü**: `sets.theme_id` → `themes.id` eşleşmesinde null ve orphan (eşleşmeyen) kayıt sayıları raporlandı.
- **`num_parts = 0`**: Bu setlerin oranı hesaplandı; bunlar muhtemelen minifig-only / promosyon setleri olduğundan **ana analizden (set büyüklüğü, parça trendleri vb.) filtrelenmesi öneriliyor.**


## 8. Filtrelenmiş çekirdek veri setini kaydet (`sets_clean.csv`)

Ana analizi çarpıtan iki grup kaydı çıkarıyoruz:

1. **`num_parts == 0`** olan setler (parça sayısı girilmemiş / sıfır).
2. **Merch (yan ürün) temaları** — gerçek LEGO seti olmayıp `Gear` (id=501) ve `Books` (id=497) alt ağaçlarındaki temalar: Key Chain, Bag and Luggage Tags, Clocks and Watches, Houseware, Magnets, Plush Toys, Posters and Art Prints, Role Play Toys and Costumes, Stationery and Office Supplies, Storage, Tabletop Games and Puzzles, Video Games and Accessories, Bags/Totes & Luggage, Clothing & Footwear, Books (Ideas/Non-fiction/Story/Activity) vb.

İki koşul birbirini tam kapsamıyor (bkz. aşağıdaki sayılar) — bazı merch ürünlerinin (peluş oyuncak, çanta süsü vb.) `num_parts > 0` olabiliyor, bu yüzden ikisi de ayrı ayrı uygulanıyor.


In [15]:
def theme_descendants(root_id: int, themes_df: pd.DataFrame) -> set:
    """root_id dahil, themes_df içindeki tüm alt temaların id kümesini döndürür."""
    ids = {root_id}
    changed = True
    while changed:
        changed = False
        newly = themes_df.loc[themes_df["parent_id"].isin(ids), "id"]
        for tid in newly:
            if tid not in ids:
                ids.add(tid)
                changed = True
    return ids

GEAR_ROOT_ID = 501   # "Gear" — anahtarlık, çanta, giyim, ev eşyası, kırtasiye vb.
BOOKS_ROOT_ID = 497  # "Books" — LEGO kitapları

merch_theme_ids = theme_descendants(GEAR_ROOT_ID, themes) | theme_descendants(BOOKS_ROOT_ID, themes)

print(f"Merch olarak işaretlenen tema sayısı: {len(merch_theme_ids)}")
themes[themes["id"].isin(merch_theme_ids)][["id", "name", "parent_id"]].sort_values("id")


Merch olarak işaretlenen tema sayısı: 24


,id,name,parent_id
236,497,Books,NaN
237,498,Technic,497.0
239,501,Gear,NaN
241,503,Key Chain,501.0
427,730,Audio and Visual Media,501.0
428,731,Bag and Luggage Tags,501.0
429,732,Clocks and Watches,501.0
430,733,Houseware,501.0
431,734,Magnets,501.0
432,735,Plush Toys,501.0


In [16]:
is_zero_parts = sets["num_parts"] == 0
is_merch = sets["theme_id"].isin(merch_theme_ids)
exclude_mask = is_zero_parts | is_merch

print(f"Toplam set                         : {len(sets):,}")
print(f"  num_parts == 0                   : {is_zero_parts.sum():,}")
print(f"  merch tema                       : {is_merch.sum():,}")
print(f"  ikisi birden (kesişim)           : {(is_zero_parts & is_merch).sum():,}")
print(f"  çıkarılan toplam (birleşim)      : {exclude_mask.sum():,}  ({exclude_mask.mean()*100:.2f}%)")
print(f"  kalan 'gerçek set'               : {(~exclude_mask).sum():,}  ({(~exclude_mask).mean()*100:.2f}%)")


Toplam set                         : 28,180
  num_parts == 0                   : 8,138
  merch tema                       : 7,754
  ikisi birden (kesişim)           : 6,611
  çıkarılan toplam (birleşim)      : 9,281  (32.93%)
  kalan 'gerçek set'               : 18,899  (67.07%)


In [17]:
sets_clean = (
    sets.loc[~exclude_mask]
    .merge(themes[["id", "name"]].rename(columns={"id": "theme_id", "name": "theme_name"}), on="theme_id", how="left")
    .loc[:, ["set_num", "name", "year", "theme_id", "theme_name", "num_parts", "img_url"]]
    .sort_values("set_num")
    .reset_index(drop=True)
)

assert sets_clean["num_parts"].gt(0).all(), "num_parts=0 kaydı sızmış"
assert not sets_clean["theme_id"].isin(merch_theme_ids).any(), "merch tema kaydı sızmış"
assert sets_clean["theme_name"].notna().all(), "eşleşmeyen theme_id var"

sets_clean.shape


(18899, 7)

In [18]:
import os

os.makedirs("../data/processed", exist_ok=True)
out_path = "../data/processed/sets_clean.csv"
sets_clean.to_csv(out_path, index=False)
print(f"Kaydedildi: {out_path}  ({len(sets_clean):,} satır)")


Kaydedildi: ../data/processed/sets_clean.csv  (18,899 satır)


**Sonuç:** `data/processed/sets_clean.csv`, `num_parts > 0` olan ve merch (Gear/Books) temalarına ait olmayan
yukarıda hesaplanan sayıda "gerçek set" kaydını içerir. Sonraki notebook'lar `sets.csv`'yi tekrar temizlemek yerine doğrudan bu dosyayı okuyabilir.


## 9. Gelecek yıl kayıtlarına karar

Bölüm 4'te görüldüğü gibi `sets.csv` bugünün yılından (2026) büyük yıl içeren 24 kayıt barındırıyor (2027 çıkışlı,
çoğu ön-duyuru/erken listeleme). Ayrıca **2026'nın kendisi de henüz tamamlanmamış bir yıl** (şu an Eylül 2026) —
bu yılın kataloğu büyümeye devam ediyor ve geç eklenen setlerin parça sayıları henüz kesinleşmemiş olabilir.

**Sorun:** Set büyüklüğü / karmaşıklık trendi gibi yıl bazlı analizlerde bu iki grup (içinde bulunulan yıl ve
gelecek yıllar) yanıltıcı: eksik/kesinleşmemiş veri, trend çizgisinin son noktasını yapay şekilde
aşağı çekebilir (veya birkaç yüksek parçalı erken duyuru varsa yukarı çekebilir).

**Karar:** Yıl bazlı trend analizleri için üst sınırı **2025 (bir önceki tam yıl)** olarak sabitliyoruz —
2026 ve sonrası, mevcut yılın kendisi değil, çünkü:

- 2026 kataloğu hâlâ dolmakta — aşağıdaki sayılar 2025'in (855 set) 2026'dan (708 set, Eylül ayında) daha
  "dolu" bir yıl olduğunu gösteriyor; 2026 yıl sonuna kadar büyümeye devam edecek.
- 2027'de tek bir ön-duyuru kaydı var — istatistiksel olarak anlamsız, tek başına bir "yıl" oluşturmuyor.
- 2025 tamamlanmış bir takvim yılı: kataloğu kapanmış, parça sayıları kesinleşmiş.

Bu kararı tek bir yerde uyguluyoruz: `sets_clean.csv`'ye trend analizlerinde kullanılmak üzere
`is_analysis_ready` (bool) kolonu ekliyoruz (`year <= YEAR_CUTOFF`). Satırlar **silinmiyor** — 2026/2027
kayıtları hâlâ gerçek, geçerli set kayıtları (örn. "bu yıl çıkanlar" gibi başka analizler için gerekli
olabilir); sadece yıl bazlı trend analizinin hangi alt kümeyi kullanması gerektiği işaretleniyor.


In [19]:
YEAR_CUTOFF = 2025  # trend analizleri için son "tam" yıl — bkz. yukarıdaki karar

print("Yıla göre kayıt sayısı (son 6 yıl):")
display(sets_clean["year"].value_counts().sort_index().tail(6))

future_or_current = sets_clean[sets_clean["year"] > YEAR_CUTOFF]
print(f"\nYEAR_CUTOFF ({YEAR_CUTOFF}) üzerindeki kayıt sayısı: {len(future_or_current):,} "
      f"({len(future_or_current) / len(sets_clean) * 100:.2f}%)")
display(future_or_current["year"].value_counts().sort_index())


Yıla göre kayıt sayısı (son 6 yıl):


year
2022    684
2023    710
2024    777
2025    855
2026    708
2027      1
Name: count, dtype: int64


YEAR_CUTOFF (2025) üzerindeki kayıt sayısı: 709 (3.75%)


year
2026    708
2027      1
Name: count, dtype: int64

In [20]:
sets_clean["is_analysis_ready"] = sets_clean["year"] <= YEAR_CUTOFF

print(sets_clean["is_analysis_ready"].value_counts())

sets_clean.to_csv("../data/processed/sets_clean.csv", index=False)
print(f"\nGüncellendi: ../data/processed/sets_clean.csv ({len(sets_clean):,} satır, "
      f"'is_analysis_ready' kolonu eklendi)")


is_analysis_ready
True     18190
False      709
Name: count, dtype: int64

Güncellendi: ../data/processed/sets_clean.csv (18,899 satır, 'is_analysis_ready' kolonu eklendi)


**Kullanım:** Yıl bazlı trend analizi yapan sonraki notebook'lar

```python
trend_df = sets_clean[sets_clean["is_analysis_ready"]]
```

satırıyla 2026/2027'nin eksik/kesinleşmemiş verisini otomatik olarak dışarıda bırakabilir; `YEAR_CUTOFF`
mantığını tekrar yazmaya gerek kalmaz.


## 10. Doğrulama: `num_parts` minifig parçalarını içeriyor mu?

**Amaç:** `sets.csv`'deki `num_parts` kolonunun minifigürlerin kendi parçalarını (kafa, gövde,
bacaklar, aksesuarlar vb.) içerip içermediğini gerçek veriyle doğrulamak. Rebrickable'ın resmi
blog açıklamasına göre içermesi bekleniyor
([*"Minifigs Suck"*](https://rebrickable.com/blog/259/minifigs-suck/)).

Bu doğrulama, Bölüm 2'deki fiyat regresyonunda `num_parts` ve `minifig_count`'u aynı anda feature
olarak kullanırsak collinearity riski olup olmadığını da netleştirecek (bkz. bölüm sonundaki karar).

**Yöntem:**

1. `num_parts > 200` olan ve envanterinde en az bir minifig bulunan setlerden 8 tanesini rastgele seç
   (`random_state=42`).
2. Her set için `inventories.csv` üzerinden **en güncel** (en yüksek `version`) `inventory_id`'yi bul —
   bazı setlerin birden fazla envanter versiyonu var (bkz. aşağıdaki sayı), `sets.csv`'deki `num_parts`
   muhtemelen en güncel versiyonu yansıtıyor.
3. `inventory_parts.csv`'den o `inventory_id`'ye ait `quantity` toplamını hesapla — **`is_spare=False`
   ve `is_spare=True` ayrı ayrı toplanıyor** (spare/yedek parçaların resmi parça sayısına dahil olup
   olmadığını da bu şekilde test ediyoruz).
4. `inventory_minifigs.csv`'den o `inventory_id`'deki minifigleri bul, her birinin `fig_num`'ı ile
   `minifigs.csv`'den kendi `num_parts`'ını çek, `quantity` ile çarp ve topla.
5. `(is_spare=False parça toplamı) + (minifig parça toplamı)`'nı `sets.csv`'deki resmi `num_parts` ile
   karşılaştır.


In [21]:
inventories = pd.read_csv(f"{RAW}/inventories.csv")
inventory_parts = pd.read_csv(f"{RAW}/inventory_parts.csv")
inventory_minifigs = pd.read_csv(f"{RAW}/inventory_minifigs.csv")
minifigs = pd.read_csv(f"{RAW}/minifigs.csv")

n_multi_version = (inventories.groupby("set_num").size() > 1).sum()
print(f"Birden fazla envanter versiyonu olan set sayısı: {n_multi_version:,} / {inventories['set_num'].nunique():,}")

# Her set_num için en güncel (en yüksek version) envanteri al
latest_inventory = (
    inventories.sort_values("version")
    .groupby("set_num", as_index=False)
    .tail(1)
)


Birden fazla envanter versiyonu olan set sayısı: 1,280 / 45,403


In [22]:
# Aday havuzu: num_parts > 200 VE envanterinde en az bir minifig bulunan setler
sets_with_minifig_inventory = set(inventory_minifigs["inventory_id"])

candidates = (
    latest_inventory[latest_inventory["id"].isin(sets_with_minifig_inventory)]
    .merge(sets[["set_num", "name", "num_parts"]], on="set_num")
    .query("num_parts > 200")
)

print(f"Aday havuzu (num_parts>200 ve minifig içeren, en güncel envanter): {len(candidates):,} set")

sample = candidates.sample(8, random_state=42).reset_index(drop=True)
sample[["set_num", "name", "num_parts"]]


Aday havuzu (num_parts>200 ve minifig içeren, en güncel envanter): 3,582 set


,set_num,name,num_parts
0,5886-1,T-Rex Hunter,476
1,80020-1,White Dragon Horse Jet,565
2,40907-1,Restaurants of the World: Mexico,326
3,60209-1,Sky Police Diamond Heist,400
4,3578-1,NHL Championship Challenge,396
5,45200-1,Moon Mission Science Kit,519
6,75642-1,Showdown with Captain Smoker,545
7,7684-1,Pig Farm & Tractor,256


In [23]:
def verify_set(inventory_id: int, official_num_parts: int) -> dict:
    ip = inventory_parts[inventory_parts["inventory_id"] == inventory_id]
    non_spare_qty = ip.loc[~ip["is_spare"], "quantity"].sum()
    spare_qty = ip.loc[ip["is_spare"], "quantity"].sum()

    im = inventory_minifigs[inventory_minifigs["inventory_id"] == inventory_id].merge(
        minifigs[["fig_num", "num_parts"]], on="fig_num", how="left"
    )
    minifig_parts_total = int((im["quantity"] * im["num_parts"]).sum())

    total_calculated = int(non_spare_qty) + minifig_parts_total
    return {
        "inventory_parts_non_spare": int(non_spare_qty),
        "inventory_parts_spare": int(spare_qty),
        "minifig_parca_toplam": minifig_parts_total,
        "toplam_hesaplanan": total_calculated,
        "fark": official_num_parts - total_calculated,
    }

results = []
for _, row in sample.iterrows():
    r = verify_set(row["id"], row["num_parts"])
    results.append({
        "set_num": row["set_num"],
        "name": row["name"],
        "num_parts": row["num_parts"],
        **r,
    })

verification = pd.DataFrame(results)[[
    "set_num", "name", "num_parts",
    "inventory_parts_non_spare", "inventory_parts_spare",
    "minifig_parca_toplam", "toplam_hesaplanan", "fark",
]]
verification


,set_num,name,num_parts,inventory_parts_non_spare,inventory_parts_spare,minifig_parca_toplam,toplam_hesaplanan,fark
0,5886-1,T-Rex Hunter,476,467,12,9,476,0
1,80020-1,White Dragon Horse Jet,565,551,0,14,565,0
2,40907-1,Restaurants of the World: Mexico,326,322,40,4,326,0
3,60209-1,Sky Police Diamond Heist,400,379,16,21,400,0
4,3578-1,NHL Championship Challenge,396,348,4,48,396,0
5,45200-1,Moon Mission Science Kit,519,508,0,11,519,0
6,75642-1,Showdown with Captain Smoker,545,528,0,17,545,0
7,7684-1,Pig Farm & Tractor,256,248,11,8,256,0


In [24]:
print(f"Tam eşleşen (fark=0) set sayısı: {(verification['fark'] == 0).sum()} / {len(verification)}")
print(f"Ortalama mutlak fark: {verification['fark'].abs().mean():.2f}")
print(f"Maksimum mutlak fark: {verification['fark'].abs().max()}")


Tam eşleşen (fark=0) set sayısı: 8 / 8
Ortalama mutlak fark: 0.00
Maksimum mutlak fark: 0


### Sonuç

**`sets.csv`'deki `num_parts`, minifig parçalarını (kafa, gövde, bacak, aksesuar vb.) içeriyor —
Rebrickable'ın blog açıklamasıyla tam uyumlu.** Örneklenen 8 setin tamamında

```
num_parts (sets.csv) == inventory_parts toplamı (is_spare=False) + minifig parça toplamı
```

**tam olarak (fark=0)** sağlandı — yaklaşık değil, birebir eşleşme. İki önemli alt bulgu:

- **Spare (yedek) parçalar `num_parts`'a dahil DEĞİL.** `is_spare=True` işaretli parçalar
  (örneklerde set başına birkaç ile birkaç düzine arası) toplama katılmadığında eşleşme sağlandı;
  dahil edildiğinde toplam `num_parts`'ı aşardı. Yani `num_parts`, kutudan çıkan *kurulum için
  gereken* parça sayısını temsil ediyor, "kutuda bulunan her parça"yı değil.
- **Minifig parçaları dahil.** Minifig başına ortalama birkaç–onlarca parça (kafa, saç/şapka,
  gövde, bacaklar, aksesuarlar) `num_parts`'a ekleniyor; bu parçalar hesaba katılmadan
  (`inventory_parts` toplamı tek başına) `num_parts`'a hiçbir örnekte ulaşılamıyor.

### Bölüm 2 (fiyat regresyonu) için sonuç: `num_parts` + `minifig_count` birlikte kullanılabilir

`num_parts` zaten minifig parçalarını içerdiği için, ayrı bir `minifig_count` (setteki minifig
**figür sayısı**, parça sayısı değil) feature'ı eklemek **doğrudan collinearity riski
oluşturmuyor** — çünkü ikisi farklı birimleri ölçüyor:

- `num_parts`: toplam parça hacmi (minifig parçaları dahil) → setin fiziksel büyüklüğü/karmaşıklığı.
- `minifig_count`: kaç *figür* var → aynı parça sayısına sahip iki set, 0 veya 5 minifig içerebilir
  (örn. bir Star Wars araç seti ile aynı boyutta bir "Creator" mimari seti) ve bu, fiyatı `num_parts`'tan
  bağımsız olarak etkileyebilir (minifigler parça başına orantısız şekilde fiyatı yükseltme eğiliminde,
  çünkü baskılı/özel kalıplı parçalar içeriyorlar).

Yine de iki feature'ın **tam bağımsız olmadığı** unutulmamalı — daha çok minifig, dolaylı olarak
biraz daha yüksek `num_parts` demek (minifig parçaları `num_parts`'ın bir alt kümesi). Regresyon
öncesi bu iki feature arasındaki korelasyon katsayısı (VIF veya basit Pearson r) ayrıca kontrol
edilmeli; güçlü bir korelasyon çıkarsa `num_parts_ex_minifigs` (`num_parts` − `minifig_parça_toplamı`)
gibi ayrıştırılmış bir feature kullanmak tercih edilebilir.
